<a href="https://colab.research.google.com/github/milesmi2019/AI_assignment_MSofCS_UEL/blob/main/BDA_SAMPLE0813.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
BDA_HDB_RESALE_PRICE_PREDICTION.py

大数据分析项目：新加坡HDB转售公寓房价预测
数据集来源：https://data.gov.sg/collections/189/view
核心模型：Gradient-Boosted Decision Trees (GBTRegressor)

更新：适用于Google Colab环境，数据集路径指向 /content/ 文件夹
优化：调整CrossValidator参数以加快执行速度，满足作业要求。
调试：增加打印语句，帮助诊断数据加载和清洗过程中的行数变化。
性能提升：扩大超参数搜索范围，并显示特征重要性。
时间优化：在交叉验证阶段使用数据采样，并用完整训练集训练最终模型。
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, year, datediff, to_date, lit, avg
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import GBTRegressor # <-- GBDT Model
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from pyspark.sql.functions import regexp_extract, when # Import for remaining_lease_years parsing

# Start PySpark Session
print("Starting PySpark Session...")
spark = SparkSession.builder \
    .appName("HDB Resale Price Prediction BDA Project") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("PySpark Session started successfully.")

# Load Datasets with PySpark
print("\nLoading datasets...")
# Updated Paths to your datasets for Google Colab /content/ directory
# Ensure these CSV files are uploaded to your Colab environment's /content/ folder
file_paths = [
    "/content/resale-flat-prices-1990-1999.csv",
    "/content/resale-flat-prices-2000-2012.csv",
    "/content/resale-flat-prices-2012-2014.csv",
    "/content/resale-flat-prices-2015-2016.csv",
    "/content/resale-flat-prices-2017-onwards.csv" # Adjusted name for the latest dataset
]

dfs = []
for path in file_paths:
    if os.path.exists(path):
        try:
            # Infer schema and header are crucial for correct loading
            temp_df = spark.read.option("header", True).csv(path, inferSchema=True)
            print(f"Loaded {path} with {temp_df.count()} rows.") # Debugging line
            dfs.append(temp_df)
        except Exception as e:
            print(f"Error loading {path}: {e}")
    else:
        print(f"Warning: {path} not found. Skipping.")

if not dfs:
    raise FileNotFoundError("No dataset files found. Please ensure CSV files are in the /content/ directory.")

# Merge all DataFrames
df_spark = dfs[0]
for df in dfs[1:]:
    df_spark = df_spark.unionByName(df, allowMissingColumns=True)

print(f"Total rows after merging all files: {df_spark.count()}") # Debugging line
df_spark.printSchema()

# Data Cleaning & Feature Engineering (PySpark)
print("\nStarting Data Cleaning and Feature Engineering...")

# Select relevant columns and cast to appropriate types
df_spark = df_spark.select(
    col("resale_price").cast("float"),
    col("floor_area_sqm").cast("float"),
    col("flat_type"),
    col("town"),
    col("storey_range"),
    col("remaining_lease"),
    col("lease_commence_date").cast("int"),
    col("month")
)
print(f"Rows after initial column selection and casting: {df_spark.count()}") # Debugging line

# Handle missing values - drop rows with any missing values for simplicity in this example
initial_rows_before_dropna = df_spark.count() # Debugging line
df_spark = df_spark.dropna()
cleaned_rows = df_spark.count()
print(f"Dropped {initial_rows_before_dropna - cleaned_rows} rows with missing values. Remaining rows: {cleaned_rows}") # Debugging line

# Check if there's enough data after initial cleaning before proceeding
if cleaned_rows == 0:
    raise ValueError("No rows left after initial data cleaning (dropna()). Please check your input data or missing values.")

# Feature Engineering
# 1. 'age_at_resale': Age of the flat at the time of resale
df_spark = df_spark.withColumn("resale_year", year(to_date(col("month"), "yyyy-MM")))
df_spark = df_spark.withColumn("age_at_resale", col("resale_year") - col("lease_commence_date"))

# 2. 'remaining_lease_years': Convert remaining_lease (e.g., "70 years 09 months") to numerical years
df_spark = df_spark.withColumn(
    "remaining_lease_years",
    when(col("remaining_lease").cast("int").isNotNull(), col("remaining_lease").cast("int"))
    .otherwise(regexp_extract(col("remaining_lease"), r'(\d+)\s+years', 1).cast("int"))
)

# 3. 'mid_storey': Average of the storey_range (e.g., "01 to 03" -> 2)
# Robust UDF for parsing storey_range
def parse_storey_range_robust(storey_range_str):
    if storey_range_str is None:
        return None
    parts = storey_range_str.split(" to ")
    if len(parts) == 2:
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            # Handle cases like "Ground Floor", "Attic" by returning None or a default
            return None
    elif storey_range_str.replace('.', '', 1).isdigit(): # Try to parse as single number (e.g., "12")
        try:
            return float(storey_range_str)
        except ValueError:
            return None
    return None # Return None for unparseable strings

parse_storey_range_udf = spark.udf.register("parse_storey_range_robust", parse_storey_range_robust)
df_spark = df_spark.withColumn("mid_storey", parse_storey_range_udf(col("storey_range")).cast("float"))

# Impute nulls in mid_storey AFTER UDF application.
# Use a common value like 0, or calculate median/mean if enough non-nulls exist.
df_spark = df_spark.withColumn("mid_storey", when(col("mid_storey").isNull(), 0.0).otherwise(col("mid_storey")))


# Drop original 'storey_range', 'remaining_lease', 'month', 'lease_commence_date', 'resale_year'
df_spark = df_spark.drop("storey_range", "remaining_lease", "month", "lease_commence_date", "resale_year")

print(f"Rows after all feature engineering: {df_spark.count()}") # Debugging line

# Categorical Feature Indexing and One-Hot Encoding
print("\nApplying StringIndexing and OneHotEncoding for categorical features...")

# Debugging: Check distinct counts of categorical features BEFORE StringIndexer
distinct_flat_types = df_spark.select("flat_type").distinct().count()
distinct_towns = df_spark.select("town").distinct().count()
print(f"Distinct 'flat_type' count before indexing: {distinct_flat_types}")
print(f"Distinct 'town' count before indexing: {distinct_towns}")

if distinct_flat_types == 0 or distinct_towns == 0:
    raise ValueError("Insufficient distinct values in 'flat_type' or 'town' after feature engineering. Please check data source and cleaning steps.")


# StringIndexer for 'flat_type'
flat_type_indexer = StringIndexer(inputCol="flat_type", outputCol="flat_type_indexed", handleInvalid="skip")
# OneHotEncoder for 'flat_type_indexed'
flat_type_encoder = OneHotEncoder(inputCol="flat_type_indexed", outputCol="flat_type_vec")

# StringIndexer for 'town'
town_indexer = StringIndexer(inputCol="town", outputCol="town_indexed", handleInvalid="skip")
# OneHotEncoder for 'town_indexed'
town_encoder = OneHotEncoder(inputCol="town_indexed", outputCol="town_vec")

# Assemble all features into a single vector
feature_cols = ['floor_area_sqm', 'age_at_resale', 'remaining_lease_years', 'mid_storey', 'flat_type_vec', 'town_vec']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Scaling numerical features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures",
                        withStd=True, withMean=False)

# Define the pipeline for preprocessing
preprocessing_pipeline = Pipeline(stages=[
    flat_type_indexer,
    flat_type_encoder,
    town_indexer,
    town_encoder,
    assembler,
    scaler
])

# Fit the preprocessing pipeline to the data
print("\nFitting preprocessing pipeline...")
preprocessing_model = preprocessing_pipeline.fit(df_spark)
df_processed = preprocessing_model.transform(df_spark)

# Select final features and label
df_final = df_processed.select("scaledFeatures", col("resale_price").alias("label"))
df_final = df_final.dropna() # Drop any rows that might have become null due to feature engineering
print(f"Final dataset rows for modeling: {df_final.count()}") # Debugging line
df_final.printSchema()

# Check if df_final is empty before splitting for modeling
if df_final.count() == 0:
    raise ValueError("Final DataFrame for modeling is empty. Cannot proceed with model training.")


# Split data into training and testing sets
print("\nSplitting data into training and testing sets (80/20)...")
(trainingData, testData) = df_final.randomSplit([0.8, 0.2], seed=42)
print(f"Training data count: {trainingData.count()}")
print(f"Test data count: {testData.count()}")

# Ensure training data is not empty
if trainingData.count() == 0:
    raise ValueError("Training data is empty. Cannot train model.")


# Model Selection: Gradient-Boosted Decision Trees (GBTRegressor)
print("\nInitializing GBTRegressor model...")
gbt = GBTRegressor(featuresCol="scaledFeatures", labelCol="label", seed=42)

# Hyperparameter Tuning using Cross-Validation for BETTER PERFORMANCE
print("\nStarting Hyperparameter Tuning with Cross-Validation for GBTRegressor (Wider Search on Sampled Data)...")

# Define parameter grid for GBTRegressor (Expanded for better performance)
paramGrid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [5, 10, 15]) \
    .addGrid(gbt.maxIter, [50, 80, 100]) \
    .addGrid(gbt.stepSize, [0.05, 0.1, 0.2]) \
    .build()

# Define the evaluator for regression tasks (RMSE is a common metric for housing prices)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

# --- Time Optimization: Sample data for Cross-Validation ---
# Sample 20% of the training data for faster Cross-Validation. Adjust fraction as needed.
sample_fraction = 0.2
sampled_trainingData = trainingData.sample(False, sample_fraction, seed=42)
print(f"Using {sampled_trainingData.count()} rows ({sample_fraction*100:.0f}%) of training data for Cross-Validation tuning.")

if sampled_trainingData.count() == 0:
    raise ValueError("Sampled training data is empty. Cannot perform cross-validation.")

# Configure CrossValidator
cv = CrossValidator(estimator=gbt,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=2, # Keeping 2 folds for reasonable execution time in Colab
                    seed=42)

# Run cross-validation to find the best model parameters on the sampled data
print("Running Cross-Validation on sampled data... This will take some time. Please be patient.")
cvModel = cv.fit(sampled_trainingData)
best_tuned_gbt_model_on_sample = cvModel.bestModel
print("Cross-Validation completed. Best GBTRegressor model parameters found from sampled data.")

# Display best hyperparameters
print(f"\nBest GBTRegressor Parameters (from sampled data): MaxDepth={best_tuned_gbt_model_on_sample._java_obj.getMaxDepth()}, "
      f"MaxIter={best_tuned_gbt_model_on_sample._java_obj.getMaxIter()}, "
      f"StepSize={best_tuned_gbt_model_on_sample._java_obj.getStepSize()}")

# --- Train Final Model on FULL Training Data using Best Parameters ---
print("\nTraining final GBTRegressor model on FULL training data using best parameters...")
final_gbt_model = GBTRegressor(
    featuresCol="scaledFeatures",
    labelCol="label",
    maxDepth=best_tuned_gbt_model_on_sample._java_obj.getMaxDepth(),
    maxIter=best_tuned_gbt_model_on_sample._java_obj.getMaxIter(),
    stepSize=best_tuned_gbt_model_on_sample._java_obj.getStepSize(),
    seed=42
)
best_gbt_model = final_gbt_model.fit(trainingData) # Train on full trainingData
print("Final GBTRegressor model trained on full training data.")


# Make predictions on the test set
print("\nMaking predictions on the test set...")
predictions = best_gbt_model.transform(testData)
predictions.select("label", "prediction", "scaledFeatures").show(5)

# Evaluate the model
rmse = evaluator.evaluate(predictions)
print(f"\nRoot Mean Squared Error (RMSE) on test data = {rmse}")

mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
mae = mae_evaluator.evaluate(predictions)
print(f"Mean Absolute Error (MAE) on test data = {mae}")

# --- Feature Importance Analysis ---
print("\nExtracting and displaying Feature Importances...")
try:
    feature_importances = best_gbt_model.featureImportances.toArray()

    flat_type_vocab = preprocessing_model.stages[0].labels
    town_vocab = preprocessing_model.stages[2].labels

    def get_expanded_feature_names(assembler_model, flat_type_labels, town_labels):
        expanded_names = []
        for col_name in assembler_model.getInputCols():
            if col_name == "flat_type_vec":
                for label in flat_type_labels:
                    expanded_names.append(f"flat_type_{label}")
            elif col_name == "town_vec":
                for label in town_labels:
                    expanded_names.append(f"town_{label}")
            else:
                expanded_names.append(col_name)
        return expanded_names

    feature_names = get_expanded_feature_names(assembler, flat_type_vocab, town_vocab)
    feature_importance_list = list(zip(feature_names, feature_importances))
    feature_importance_list.sort(key=lambda x: x[1], reverse=True)

    print("\nMost Important Features:")
    for feature, importance in feature_importance_list[:10]: # Print top 10
        print(f"- {feature}: {importance:.6f}")

except Exception as e:
    print(f"Could not retrieve feature importances: {e}. Ensure 'best_gbt_model' is a tree-based model.")


# Compare with a simpler model (e.g., Linear Regression) for verification
print("\nTraining and evaluating a Linear Regression model for comparison...")
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="scaledFeatures", labelCol="label", maxIter=10, regParam=0.3, elasticNetParam=0.8, seed=42)
lr_model = lr.fit(trainingData) # Train LR on full training data
lr_predictions = lr_model.transform(testData)
lr_rmse = evaluator.evaluate(lr_predictions)
lr_mae = mae_evaluator.evaluate(lr_predictions)

print(f"Linear Regression RMSE on test data = {lr_rmse}")
print(f"Linear Regression MAE on test data = {lr_mae}")

if rmse < lr_rmse:
    print("\nGBTRegressor performed better than Linear Regression (lower RMSE).")
else:
    print("\nLinear Regression performed better or similarly to GBTRegressor.")


# Visualization of Predictions vs. Actual (Sampling for plotting)
print("\nGenerating scatter plot of Actual vs. Predicted prices...")
predictions_df = predictions.select("label", "prediction").sample(False, 0.1, seed=42).toPandas()

plt.figure(figsize=(10, 6))
plt.scatter(predictions_df["label"], predictions_df["prediction"], alpha=0.3)
plt.plot([predictions_df["label"].min(), predictions_df["label"].max()],
         [predictions_df["label"].min(), predictions_df["label"].max()],
         'r--', lw=2, label='Perfect Prediction Line')
plt.xlabel("Actual Resale Price")
plt.ylabel("Predicted Resale Price")
plt.title("Actual vs. Predicted Resale Prices (GBTRegressor)")
plt.grid(True)
plt.legend()
plt.show()


# Example of making a new prediction (demonstrates model deployment readiness)
print("\nDemonstrating single prediction for a new HDB flat...")

# Get the StringIndexer and OneHotEncoder models from the fitted pipeline
flat_type_indexer_model = preprocessing_model.stages[0]
town_indexer_model = preprocessing_model.stages[2]
scaler_model = preprocessing_model.stages[5]

def predict_hdb_price(floor_area_sqm, flat_type, town, remaining_lease_years, mid_storey, age_at_resale):
    """
    Predicts the resale price of an HDB flat using the trained GBTRegressor model.

    Args:
        floor_area_sqm (float): Floor area in square meters.
        flat_type (str): Type of flat (e.g., '3 ROOM', '4 ROOM', '5 ROOM').
        town (str): Town name (e.g., 'ANG MO KIO', 'JURONG WEST').
        remaining_lease_years (int): Remaining lease in years.
        mid_storey (float): Midpoint of the storey range (e.g., 2.0 for '01 to 03').
        age_at_resale (int): Age of the flat at the time of resale.

    Returns:
        float: Predicted resale price.
    """
    try:
        new_data_row = spark.createDataFrame([(
            floor_area_sqm,
            flat_type,
            town,
            remaining_lease_years,
            mid_storey,
            age_at_resale
        )], schema=[
            "floor_area_sqm",
            "flat_type",
            "town",
            "remaining_lease_years",
            "mid_storey",
            "age_at_resale"
        ])

        # Apply preprocessing stages
        new_data_indexed = flat_type_indexer_model.transform(new_data_row)
        new_data_indexed = town_indexer_model.transform(new_data_indexed)
        new_data_encoded = flat_type_encoder.transform(new_data_indexed)
        new_data_encoded = town_encoder.transform(new_data_encoded)
        assembled_features = assembler.transform(new_data_encoded)
        scaled_features = scaler_model.transform(assembled_features)

        # Make prediction with the final best GBT model
        prediction_df = best_gbt_model.transform(scaled_features)
        predicted_price = prediction_df.collect()[0]["prediction"]
        return predicted_price
    except Exception as e:
        print(f"Error during prediction: {e}")
        return None

# Example usage of the prediction function
area = 95.0
flat = "4 ROOM"
town_name = "QUEENSTOWN"
lease_rem = 70
storey = 10.5
age = 20

predicted_price = predict_hdb_price(area, flat, town_name, lease_rem, storey, age)

if predicted_price is not None:
    print(f"\nPredicted resale price for a {flat} flat in {town_name} "
          f"(Area: {area} sqm, Lease Rem: {lease_rem} yrs, Storey: {storey}, Age: {age} yrs): "
          f"SGD {predicted_price:,.2f}")
else:
    print("\nCould not make a prediction. Please check input values and data consistency.")

# Stop Spark Session
print("\nStopping PySpark Session.")
spark.stop()
print("PySpark Session stopped.")

Starting PySpark Session...
PySpark Session started successfully.

Loading datasets...
Loaded /content/resale-flat-prices-1990-1999.csv with 287196 rows.
Loaded /content/resale-flat-prices-2000-2012.csv with 369651 rows.
Loaded /content/resale-flat-prices-2012-2014.csv with 52203 rows.
Loaded /content/resale-flat-prices-2015-2016.csv with 37153 rows.
Loaded /content/resale-flat-prices-2017-onwards.csv with 213040 rows.
Total rows after merging all files: 959243
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- remaining_lease: string (nullable = true)


Starting Data Cleaning and Feature Engineering...
Rows 